In [10]:
import cv2
import numpy as np
import glob
import os
from pathlib import Path

def convert_jpg_to_uyvy(input_path, output_path):
    """
    Convert a JPG image to UYVY format and save as raw file.
    
    Args:
        input_path: Path to input JPG file
        output_path: Path to save UYVY raw file
    """
    # Read image with OpenCV (this loads as BGR)
    bgr = cv2.imread(input_path)
    if bgr is None:
        print(f"Failed to read image: {input_path}")
        return False
    
    # Convert BGR to YUV (OpenCV uses YUV 4:2:2)
    yuv = cv2.cvtColor(bgr, cv2.COLOR_BGR2YUV)
    
    height, width = yuv.shape[:2]
    
    # Prepare UYVY array (width * 2 because each pixel pair needs 4 bytes)
    uyvy = np.zeros((height, width * 2), dtype=np.uint8)
    
    # For each pair of pixels
    for y in range(height):
        for x in range(0, width, 2):
            if x + 1 >= width:  # Handle odd width
                # For last pixel if width is odd
                u = yuv[y, x, 1]
                y0 = yuv[y, x, 0]
                v = yuv[y, x, 2]
                
                uyvy[y, x*2] = u      # U
                uyvy[y, x*2 + 1] = y0  # Y0
                uyvy[y, x*2 + 2] = v   # V
                uyvy[y, x*2 + 3] = y0  # Y1 (duplicate last Y)
            else:
                # Get U, Y0, V from first pixel
                u = yuv[y, x, 1]
                y0 = yuv[y, x, 0]
                v = yuv[y, x, 2]
                # Get Y1 from second pixel
                y1 = yuv[y, x + 1, 0]
                
                # Pack as UYVY
                uyvy[y, x*2] = u      # U
                uyvy[y, x*2 + 1] = y0  # Y0
                uyvy[y, x*2 + 2] = v   # V
                uyvy[y, x*2 + 3] = y1  # Y1
    
    # Save as raw file
    uyvy.tofile(output_path)
    return True

def convert_folder(input_folder, output_folder):
    """
    Convert all JPG images in a folder to UYVY format.
    
    Args:
        input_folder: Path to folder containing JPG images
        output_folder: Path to save UYVY files
    """
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    # print full path to input and output folders
    print(f"Input folder: {os.path.abspath(input_folder)}")
    print(f"Output folder: {os.path.abspath(output_folder)}")
    
    # Get all jpg files
    jpg_files = glob.glob(os.path.join(input_folder, "*.jpg"))
    total_files = len(jpg_files)
    
    print(f"Found {total_files} JPG files to convert")
    
    success_count = 0
    for i, jpg_path in enumerate(jpg_files, 1):
        # Create output path
        filename = Path(jpg_path).stem
        output_path = os.path.join(output_folder, f"{filename}.raw")
        
        # Convert file
        if convert_jpg_to_uyvy(jpg_path, output_path):
            success_count += 1
        
        # Progress update
        if i % 10 == 0:
            print(f"Processed {i}/{total_files} files")
    
    print(f"\nConversion complete!")
    print(f"Successfully converted {success_count}/{total_files} files")
    print(f"Output files saved to: {output_folder}")

# Example usage
if __name__ == "__main__":
    # Replace these paths with your actual paths
    input_folder = "./dataset/images/val"
    output_folder = "./dataset_uyvy/images/val"
    
    convert_folder(input_folder, output_folder)

Input folder: /home/daniel/Documents/GitHub/paparazzi/SmallConvNetwork/dataset/images/val
Output folder: /home/daniel/Documents/GitHub/paparazzi/SmallConvNetwork/dataset_uyvy/images/val
Found 63 JPG files to convert
Processed 10/63 files
Processed 20/63 files
Processed 30/63 files
Processed 40/63 files
Processed 50/63 files
Processed 60/63 files

Conversion complete!
Successfully converted 63/63 files
Output files saved to: ./dataset_uyvy/images/val
